# Hateful Memes: Does Fusion Beat Unimodal? (Colab / Kaggle GPU)
Runtime -> Change runtime type -> **T4 GPU** (Kaggle: Settings -> Accelerator -> GPU T4, Internet ON).

**Everyone runs sections 1 and 2. After that, run only the sections with your name** (see `TEAM_PLAN.md`).
Owners: **A** = baselines, **B** = data / early fusion / CLIP features, **C** = analysis, **Mohit** = CLIP fusion.
After training, save this notebook with outputs (File -> Download .ipynb) and add it to your results PR.

In [ ]:
# 1. Get the code (safe to re-run: clones once, afterwards syncs to the latest code on GitHub)
import os
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"
os.chdir(BASE)
if not os.path.isdir("repo"):
    !git clone https://github.com/sharma-mohit-2005/hateful-memes-fusion.git repo
%cd {BASE}/repo
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# 1b. (Colab, recommended) keep the dataset and CLIP features on Google Drive, so a disconnect
# does not mean downloading 4.5 GB again. Results stay in the repo and are backed up to Drive (cell below).
%cd {BASE}/repo
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/hateful_memes"
!mkdir -p {D}/data {D}/features {D}/results_backup
!rm -rf data features; ln -s {D}/data data; ln -s {D}/features features
# restore results saved before a disconnect (does nothing the first time)
!mkdir -p results && cp -rn {D}/results_backup/. results/ 2>/dev/null; ls results/runs 2>/dev/null

### Backup: run this after every training cell (Colab), so a disconnect does not lose your results

In [ ]:
!cp -r results/. {D}/results_backup/ && ls {D}/results_backup/runs

## 2. Data — EVERYONE (~10-20 min, safe to re-run: already downloaded files are skipped). `eda.py` — Member B only

In [ ]:
!python src/download_data.py
!python src/prepare_splits.py

In [ ]:
!python src/eda.py   # Member B

## 3. Unimodal baselines — Member A (M1 ~20 min, M2 ~40 min, M4 1 min)

In [ ]:
!python src/train_unimodal.py --modality text --seeds 0 1 2

In [ ]:
!python src/train_unimodal.py --modality image --seeds 0 1 2

In [ ]:
!python src/late_fusion.py --text_run M1_bert_text --image_run M2_resnet_image

## 4. Early fusion M3 — Member B (~1.5 h)

In [ ]:
!python src/train_unimodal.py --modality both --seeds 0 1 2 --batch 16

## 5. CLIP features — Member B, Mohit and Member C (each runs it once in their own Colab, ~10 min)
Features are not committed to GitHub, so whoever needs them extracts them.

In [ ]:
!python src/extract_clip.py --tag clip_l14

## 6. CLIP linear probe + fusion ablation — Mohit (~15 min; Member C also runs this cell for the modality test)

In [ ]:
!python src/linear_probe.py --features clip_l14
for f in ["image_only", "text_only", "concat", "product", "concat_product", "cross", "gated", "attention"]:
    !python src/train_head.py --fusion {f} --features clip_l14 --seeds 0 1 2
!python src/train_head.py --fusion concat --features clip_l14 --modality_dropout 0.3 --seeds 0 1 2
!python src/late_fusion.py --text_run C_clip_l14_text_only --image_run C_clip_l14_image_only --prefix C_late

## 7. Text removal (inpainting) experiment — Member C (~40 min)

In [ ]:
!python src/analysis/inpaint_text.py
!python src/extract_clip.py --img_root data/inpainted --tag clip_l14_inpainted
for f in ["image_only", "concat", "cross"]:
    !python src/train_head.py --fusion {f} --features clip_l14_inpainted --seeds 0 1 2

## 8. Final analysis — only AFTER everyone's results PRs are merged
Run the sync cell first so you have everyone's predictions.

In [ ]:
# brings everyone's merged results + latest code (your own unmerged results are kept)
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1

In [ ]:
# Member C: benign-confounder analysis + modality reliance test (needs sections 5, 6, 7 run in this Colab)
!python src/analysis/confounders.py --split test --sim_features clip_l14_inpainted --show_run C_clip_l14_cross
!python src/analysis/modality_ablation.py --split test

In [ ]:
!python src/analysis/error_examples.py --run C_clip_l14_cross --split test   # Member A

In [ ]:
# Mohit: final results table + bar chart
!python src/analysis/make_tables.py
from IPython.display import Markdown, Image, display
display(Markdown(open("results/summary.md").read()))
display(Image("results/figures/test_auroc.png"))

## 9. Save your results: download results.zip, unzip into your local repo, open a results PR

In [ ]:
!zip -qr results.zip results -x '*.pt' -x 'results/figures/eda_samples.png'